# Cuaderno 03: Generación de Variables Temporales

En esta fase aplicaremos técnicas de generación de variables predictivas para captar el historial deportivo de los equipos. 

Para evitar cualquier filtración de datos, todas las variables se calcularán:
1. Agrupando por temporada (reiniciando los contadores cada año).
2. Utilizando información estrictamente anterior al inicio del partido.
3. Adaptando las ventanas de cálculo si nos encontramos al inicio de la temporada.

In [1]:
import pandas as pd
import sys
import os

# Añadimos el directorio raíz a la lista de rutas de Python para poder importar desde 'src'
sys.path.append(os.path.abspath('..'))

%load_ext autoreload
%autoreload 2

from src.features import calcular_acumulados_temporada

df = pd.read_csv('../data/interim/train_limpio.csv')
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)


## 1. Acumulados de la Temporada
Calculamos el rendimiento general del equipo desde la Jornada 1 hasta el momento actual del partido. Extraemos:
* Puntos acumulados del equipo local y visitante.
* Diferencia de goles acumulada (Goles a Favor - Goles en Contra) del local y visitante.

In [2]:
# Aplicamos la función
df_features = calcular_acumulados_temporada(df)

# Comprobación visual, para ver que se inicie a 0 y que vaya acumulando correctamente a lo largo de la temporada 
comprobacion = df_features[
    (df_features['anio_fin_temporada'] == 2023) & 
    (df_features['local'] == 'Almeria')
]
comprobacion[['fecha', 'local', 'pts_acum_local', 'dif_goles_acum_local']].head(3)

,fecha,local,pts_acum_local,dif_goles_acum_local
16742,2022-08-14,Almeria,0.0,0.0
16781,2022-08-27,Almeria,1.0,-1.0
16829,2022-09-12,Almeria,4.0,-1.0


## 2. Rachas Recientes
Calculamos cómo llega cada equipo al partido analizando sus resultados más recientes. Generamos variables para los últimos 3 y 5 partidos, extrayendo:
* Puntos sumados en esa ventana.
* Promedio de goles a favor y en contra.

Utilizamos ventanas deslizantes (`rolling`) asegurando aislar cada temporada y permitiendo calcular la media aunque no se haya alcanzado el tamaño máximo de la ventana (`min_periods=1`).

In [3]:
from src.features import calcular_rachas_recientes

# 1. Calculamos las rachas de los últimos 3 partidos
# Le pasamos el df_features que ya tiene los cálculos del bloque 1
df_features = calcular_rachas_recientes(df_features, ventana=3)

# 2. Calculamos las rachas de los últimos 5 partidos
df_features = calcular_rachas_recientes(df_features, ventana=5)

# 3. Comprobación visual
columnas_ver = [
    'fecha', 
    'local', 
    'visitante', 
    'pts_ultimos_3_local',  
    'pts_ultimos_3_visitante',
    'gf_media_3_local',     
    'gf_media_3_visitante',     
    'gc_media_3_local',     
    'gc_media_3_visitante'
]
comprobacion_racha = df_features[
    (df_features['anio_fin_temporada'] == 2023) & 
    (df_features['local'] == 'Almeria')
]

comprobacion_racha[columnas_ver].head(5)

,fecha,local,visitante,pts_ultimos_3_local,pts_ultimos_3_visitante,gf_media_3_local,gf_media_3_visitante,gc_media_3_local,gc_media_3_visitante
16742,2022-08-14,Almeria,Real Madrid,0.0,0.0,0.000000,0.000000,0.000000,0.000000
16781,2022-08-27,Almeria,Sevilla,1.0,1.0,1.000000,1.000000,1.500000,1.500000
16829,2022-09-12,Almeria,Osasuna,4.0,6.0,1.000000,1.333333,1.000000,0.666667
16876,2022-10-08,Almeria,Vallecano,0.0,6.0,0.000000,2.000000,2.000000,1.666667
16925,2022-10-20,Almeria,Girona,3.0,1.0,1.333333,1.666667,2.666667,2.666667


## 3. Rendimiento Específico
Extraemos el rendimiento contextual aislando el factor campo. 

Calculamos:
* Promedio de puntos que consigue el equipo Local cuando juega en su estadio.
* Promedio de puntos que consigue el equipo Visitante cuando juega a domicilio.

In [4]:
from src.features import calcular_rendimiento_local_visitante

# Aplicamos la función
df_features = calcular_rendimiento_local_visitante(df_features)

# Comprobación visual
columnas_ver = ['fecha', 'visitante', 'promedio_pts_fuera']

comprobacion_madrid_fuera = df_features[
    (df_features['anio_fin_temporada'] == 2023) & 
    (df_features['visitante'] == 'Almeria')
]

comprobacion_madrid_fuera[columnas_ver].head(4)

,fecha,visitante,promedio_pts_fuera
16767,2022-08-22,Almeria,0.000000
16819,2022-09-05,Almeria,1.000000
16837,2022-09-17,Almeria,0.500000
16847,2022-09-30,Almeria,0.333333


## 4. Tratamiento de Valores Nulos y Exportación a la zona Processed

Al generar variables de rachas e históricos mediante ventanas temporales, es matemáticamente inevitable que los primeros partidos de cada equipo carezcan de historial previo, lo que genera valores nulos en el dataset. 

Dado que un equipo en la primera jornada de liga efectivamente tiene 0 puntos acumulados y un promedio de 0 goles, la imputación más rigurosa y fiel a la realidad deportiva es rellenar estos vacíos con el valor 0. De esta forma, evitamos eliminar los primeros partidos de cada temporada y mantenemos el 100% de la varianza original.

Finalmente, exportaremos este dataset enriquecido a la carpeta de datos definitivos (`data/processed/`), dejándolo preparado para la fase final de aprendizaje automático.

In [ ]:

# 1. Rellenamos los historiales vacíos de los primeros partidos con 0 y redondeamos a 2 decimales para que no quede un montón de números con decimales raros.
df_features = df_features.fillna(0)
df_features = df_features.round(2)

# 2. Comprobamos que no se ha perdido ni una sola fila del dataset original
print(f"Filas originales: {len(df)}")
print(f"Filas finales: {len(df_features)}")

# 3. Guardamos en la carpeta definitiva
df_features.to_csv('../data/processed/train_features.csv', index=False)



C:\Users\movil\AppData\Local\Temp\ipykernel_44568\3526700961.py:3: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_features = df_features.round(2)


Filas originales: 17480
Filas finales: 17480
¡Dataset final enriquecido y guardado con éxito!
